# 01 EDA

Exploratory analysis for electricity prices, carbon intensity, demand, weather, and workload signals.

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data.load import create_database_engine
from src.data.quality import run_quality_checks, print_quality_report

In [3]:
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

True

In [4]:
engine = create_database_engine(os.environ["DATABASE_URL"])

results = run_quality_checks(engine)
print_quality_report(results)

PASS: table_coverage
  {'table_name': 'electricity_prices', 'row_count': 30623, 'min_timestamp_utc': datetime.datetime(2022, 12, 31, 23, 0, tzinfo=datetime.timezone.utc), 'max_timestamp_utc': datetime.datetime(2026, 6, 30, 21, 0, tzinfo=datetime.timezone.utc)}
  {'table_name': 'hourly_electricity_mix', 'row_count': 379327, 'min_timestamp_utc': datetime.datetime(2023, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), 'max_timestamp_utc': datetime.datetime(2026, 4, 30, 21, 0, tzinfo=datetime.timezone.utc)}
  {'table_name': 'weather_observations', 'row_count': 367776, 'min_timestamp_utc': datetime.datetime(2023, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), 'max_timestamp_utc': datetime.datetime(2026, 6, 30, 23, 0, tzinfo=datetime.timezone.utc)}
PASS: duplicate_source_ids
FAIL: missing_price_hours
  {'region': 'FR', 'missing_price_hours': 24}
FAIL: missing_mix_hours
  {'region': 'FR', 'scope': 'national', 'missing_mix_hours': 3}
  {'region': 'Auvergne-Rhône-Alpes', 'scope': 'regional', 'missing_mix

In [1]:
import json
from pathlib import Path

In [11]:
report = json.loads(Path("../reports/metrics/pipeline_health.json").read_text())

In [12]:
print("STATUS:", report["status"])
print("CRITICAL:", report["critical_issue_count"])
print("WARNINGS:", report["warning_count"])
print()

STATUS: warn
CRITICAL: 0
WARNINGS: 9



In [13]:
for name, source in report["sources"].items():
    if source["critical_issues"] or source["warnings"]:
        print(f"{name}")
        print("  latest:", source["max_timestamp_utc"])
        print("  freshness lag days:", source["freshness_lag_days"])
        print("  critical:", source["critical_issues"])
        print("  warnings:", source["warnings"])
        print()

electricity_prices
  latest: 2026-08-10T21:00:00+00:00
  freshness lag days: -0.02
  critical: []
  warnings: ['missing_hour_count:24']

hourly_electricity_mix
  latest: 2026-08-10T23:00:00+00:00
  freshness lag days: -0.1
  critical: []
  warnings: ['missing_hour_count:65', 'null_count:consumption_mwh:30', 'null_count:nuclear_mwh:30', 'null_count:wind_mwh:30', 'null_count:solar_mwh:30', 'null_count:hydro_mwh:30', 'null_count:bioenergy_mwh:30']

modeling_price_features
  latest: 2026-08-10T17:00:00+00:00
  freshness lag days: 0.15
  critical: []
  warnings: ['missing_hour_count:898']

